# Conditional clade topology sampling: comparing gradient estimators

This notebook demonstrates TreeFlow's **conditional clade distribution** (a
*subsplit Bayesian network*, SBN) over rooted tree topologies, and compares
several gradient estimators for learning its parameters by stochastic gradient
descent.

## Background

A **clade** is a subset of the taxa (the leaves under a node of a rooted tree).
A **subsplit** is the ordered pair of child clades produced when an internal node
splits its parent clade in two. A conditional clade distribution factorises the
probability of a rooted topology over its internal nodes:

$$ P(\tau) = \prod_{v \in \text{internal}(\tau)} P(\text{subsplit}_v \mid \text{clade}_v). $$

Each conditional $P(\cdot \mid C)$ is a categorical over the subsplits of clade
$C$, parametrised by a softmax over a vector of logits. The whole distribution is
therefore differentiable in a single flat logit vector, while *sampling* a
topology is discrete.

## The experiment

We pick a random **target** distribution $p$ (a CCD with known logits) and fit a
**variational** CCD $q$ to it by minimising the reverse KL divergence
$\mathrm{KL}(q \Vert p) = \mathbb{E}_{\tau \sim q}[\log q(\tau) - \log p(\tau)]$.
For small taxon sets we can enumerate every topology, so we have an **exact**
KL (and its exact gradient) to use as ground truth. We compare:

* **Exact gradient** &mdash; differentiate the enumerated KL (ground truth).
* **Score function / REINFORCE** &mdash; with and without a leave-one-out baseline.
* **VIMCO** &mdash; the multi-sample importance-weighted bound.
* **Straight-through Gumbel-Softmax** &mdash; the relaxation / '1-0 probability
  gradient' sampler.


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from treeflow import DEFAULT_FLOAT_DTYPE_TF as FLOAT
from treeflow.conditional_clade.support import ConditionalCladeSupport
from treeflow.conditional_clade.distribution import ConditionalCladeDistribution
from treeflow.conditional_clade.estimators import (
    score_function_surrogate,
    leave_one_out_baseline,
    vimco_surrogate,
    sample_relaxed_cost,
)

tf.random.set_seed(0)
np.set_printoptions(precision=3, suppress=True)

## 1. Build the support and a target distribution

We work with `N_TAXA = 5` taxa. The support enumerates every splittable clade and
every subsplit; the number of distinct rooted topologies is the double factorial
$(2n-3)!!$.


In [ ]:
N_TAXA = 5
support = ConditionalCladeSupport(N_TAXA, taxon_set=[f't{i}' for i in range(N_TAXA)])

print(f'taxa:                {support.taxon_count}')
print(f'splittable clades:   {support.parent_clade_count}')
print(f'subsplits (logits):  {support.subsplit_count}')
print(f'rooted topologies:   {support.topology_count()}')

The subsplit support also exposes **binary-vector features** for every subsplit
(the membership vectors of the parent and two child clades, concatenated). These
are the hook for an *embedding-based* parametrisation: instead of one free logit
per subsplit, a model could map these features to logits, which scales to taxon
sets where enumerating every subsplit is infeasible.


In [ ]:
features = support.subsplit_feature_matrix()
print('subsplit feature matrix shape (M, 3n):', features.shape)
print('first subsplit features [parent | child1 | child2]:')
print(features[0].reshape(3, N_TAXA))

In [ ]:
# A random target CCD. Scaling the logits makes the target reasonably peaked.
rng = np.random.default_rng(1)
target_logits = tf.constant(2.0 * rng.standard_normal(support.subsplit_count), FLOAT)
p = ConditionalCladeDistribution(support, target_logits)

target_probs = p.enumerate_probs().numpy()
order = np.argsort(target_probs)[::-1]
print('target distribution over topologies sums to', target_probs.sum())
print('top-5 topology probabilities:', target_probs[order[:5]])

plt.figure(figsize=(7, 3))
plt.bar(range(len(target_probs)), np.sort(target_probs)[::-1])
plt.xlabel('topology (sorted by probability)')
plt.ylabel('p(topology)')
plt.title(f'Target CCD over {support.topology_count()} rooted topologies')
plt.tight_layout(); plt.show()

## 2. Sampling a topology

Sampling walks down from the root clade, drawing a subsplit at each splittable
clade, and converts the result to TreeFlow's `parent_indices` topology encoding
(leaves `0..n-1`, internal nodes `n..2n-2`, root last).


In [ ]:
sample_rng = np.random.default_rng(7)
topology = p.sample_topology(sample_rng)
print('parent_indices:', topology.parent_indices)
print('child_indices:\n', topology.child_indices)

# Recover the subsplit assignment and show the clades as taxon sets.
from treeflow.conditional_clade.clade import clade_taxa
assignment = support.parent_indices_to_assignment(topology.parent_indices)
for parent, subsplit in sorted(assignment.items(), key=lambda kv: -bin(kv[0]).count('1')):
    print(f'{clade_taxa(parent)} -> {clade_taxa(subsplit.child1)} | {clade_taxa(subsplit.child2)}')

## 3. Estimators

Each estimator returns a gradient of $\mathrm{KL}(q\Vert p)$ with respect to the
variational logits. We use the *surrogate-loss* convention: build a scalar whose
`tf.GradientTape` gradient is the estimator, then read it off.


In [ ]:
def make_q(seed):
    init = np.random.default_rng(seed).standard_normal(support.subsplit_count)
    logits = tf.Variable(tf.constant(init, FLOAT))
    return ConditionalCladeDistribution(support, logits), logits

# Pre-fetch the (constant) target conditional log-probs once.
log_p_cond = p.conditional_log_probs()

def sample_log_probs(q, flat_indices):
    flat = tf.constant(flat_indices, tf.int32)
    log_q = q.log_prob_from_flat_indices(flat)
    log_p = tf.reduce_sum(tf.gather(log_p_cond, flat), axis=-1)
    return log_q, log_p

In [ ]:
def grad_exact(q, logits, rng, batch_size=None):
    with tf.GradientTape() as tape:
        kl = q.exact_kl_divergence(p)
    return tape.gradient(kl, logits)

def grad_score(q, logits, rng, batch_size=64, baseline_state=None):
    flat = q.sample_flat_index_batch(batch_size, rng)
    with tf.GradientTape() as tape:
        log_q, log_p = sample_log_probs(q, flat)
        cost = log_q - log_p
        # constant baseline = batch mean (a simple control variate)
        baseline = tf.reduce_mean(tf.stop_gradient(cost))
        surrogate = score_function_surrogate(cost, log_q, baseline)
    return tape.gradient(surrogate, logits)

def grad_score_loo(q, logits, rng, batch_size=64):
    flat = q.sample_flat_index_batch(batch_size, rng)
    with tf.GradientTape() as tape:
        log_q, log_p = sample_log_probs(q, flat)
        cost = log_q - log_p
        baseline = leave_one_out_baseline(cost)
        surrogate = score_function_surrogate(cost, log_q, baseline)
    return tape.gradient(surrogate, logits)

def grad_vimco(q, logits, rng, batch_size=64):
    flat = q.sample_flat_index_batch(batch_size, rng)
    with tf.GradientTape() as tape:
        log_q, log_p = sample_log_probs(q, flat)
        log_weights = log_p - log_q
        loss = -vimco_surrogate(log_q, log_weights)  # maximise the bound
    return tape.gradient(loss, logits)

def grad_straight_through(q, logits, rng, batch_size=64, temperature=0.5):
    with tf.GradientTape() as tape:
        costs = []
        for _ in range(batch_size):
            s = sample_relaxed_cost(q, p, temperature=temperature, gumbel=True)
            costs.append(s.log_q - s.log_p)
        loss = tf.add_n(costs) / len(costs)
    return tape.gradient(loss, logits)

ESTIMATORS = {
    'exact': grad_exact,
    'score function': grad_score,
    'score + leave-one-out': grad_score_loo,
    'VIMCO': grad_vimco,
    'straight-through GS': grad_straight_through,
}

## 4. Gradient quality at a fixed point

Before training, we compare each estimator against the exact gradient at a fixed
set of logits. We report:

* **cosine similarity** to the exact gradient (direction; the score-function and
  VIMCO estimators are unbiased, so this approaches 1 as we average more
  samples; straight-through is biased), and
* **per-estimate variance** (a proxy for how noisy a single SGD step is).


In [ ]:
q0, logits0 = make_q(seed=2)
exact = ESTIMATORS['exact'](q0, logits0, None).numpy()

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

rng = np.random.default_rng(10)
n_reps = 200
rows = []
for name, fn in ESTIMATORS.items():
    if name == 'exact':
        continue
    grads = np.stack([fn(q0, logits0, rng, batch_size=16).numpy() for _ in range(n_reps)])
    mean_grad = grads.mean(0)
    variance = grads.var(0).sum()              # total variance across logits
    bias = np.linalg.norm(mean_grad - exact)   # 0 for unbiased estimators (in the limit)
    rows.append((name, cosine(mean_grad, exact), variance, bias))

print(f"{'estimator':24s} {'cos(mean, exact)':>16s} {'variance':>12s} {'||bias||':>10s}")
for name, cos, var, bias in rows:
    print(f'{name:24s} {cos:16.3f} {var:12.3f} {bias:10.3f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
names = [r[0] for r in rows]
ax[0].bar(names, [r[2] for r in rows], color='tab:orange')
ax[0].set_ylabel('total gradient variance'); ax[0].set_yscale('log')
ax[0].set_title('Per-step gradient variance (batch=16)')
ax[0].tick_params(axis='x', rotation=30)
ax[1].bar(names, [r[1] for r in rows], color='tab:green')
ax[1].set_ylabel('cosine to exact gradient'); ax[1].set_ylim(0, 1.05)
ax[1].set_title('Direction agreement of the mean estimate')
ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

The unbiased estimators (score function, leave-one-out, VIMCO) point in the exact
direction on average; the leave-one-out baseline and VIMCO cut the variance of
plain REINFORCE substantially. Straight-through Gumbel-Softmax is biased &mdash; its
mean does not match the exact gradient &mdash; but has low variance, which often
makes it competitive in practice.


## 5. Optimisation

Now we actually run SGD with each estimator, tracking the exact
$\mathrm{KL}(q\Vert p)$ (which we can compute because the taxon set is small).


In [ ]:
def train(grad_fn, steps=400, lr=0.05, batch_size=32, seed=2, eval_every=5):
    q, logits = make_q(seed)
    opt = tf.optimizers.Adam(lr)
    rng = np.random.default_rng(100)
    history = []
    for step in range(steps):
        grad = grad_fn(q, logits, rng, batch_size=batch_size)
        opt.apply_gradients([(grad, logits)])
        if step % eval_every == 0:
            history.append((step, float(q.exact_kl_divergence(p).numpy())))
    return np.array(history)

histories = {}
for name, fn in ESTIMATORS.items():
    bs = 32 if name != 'straight-through GS' else 16
    histories[name] = train(fn, steps=400, lr=0.05, batch_size=bs)
    print(f'{name:24s} final KL = {histories[name][-1, 1]:.4f}')

In [ ]:
plt.figure(figsize=(8, 5))
for name, hist in histories.items():
    style = '--' if name == 'exact' else '-'
    plt.plot(hist[:, 0], hist[:, 1], style, label=name)
plt.yscale('log')
plt.xlabel('SGD step')
plt.ylabel('exact KL(q || p)')
plt.title('Fitting a conditional clade distribution with different gradient estimators')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Summary

* TreeFlow now has a differentiable **conditional clade / subsplit** distribution
  over rooted topologies, with exact enumeration, sampling, log-probabilities and
  exact KL for small taxon sets, plus conversion to the standard `parent_indices`
  topology encoding.
* The representation is **embedding-ready**: clades and subsplits expose binary
  membership vectors, so the per-subsplit logits can be produced by a model
  rather than stored explicitly.
* For learning the parameters through discrete topology samples:
  * **Score function (REINFORCE)** is unbiased but high variance; a **leave-one-out**
    baseline and **VIMCO** reduce that variance markedly.
  * **Straight-through Gumbel-Softmax** (the '1-0 probability gradient' sampler)
    is biased but low variance and cheap.
* All estimators are compared against the **exact** enumerated gradient, available
  here only because the taxon set is small &mdash; exactly the regime this module is
  designed to support.
